# Mchakato wa Binadamu katika Mzunguko na Microsoft Agent Framework

## 🎯 Malengo ya Kujifunza

Katika daftari hili, utajifunza jinsi ya kutekeleza mifumo ya **binadamu katika mzunguko** ukitumia Microsoft Agent Framework `RequestInfoExecutor`. Mtindo huu wenye nguvu unakuwezesha kusitisha mchakato wa AI kukusanya habari kutoka kwa binadamu, kufanya maajenti wako kuwa ya kuingiliana na kuwapa binadamu udhibiti juu ya maamuzi muhimu.

## 🔄 Je, Binadamu Katika Mzunguko ni Nini?

**Binadamu katika mzunguko (HITL)** ni mtindo wa usanifu ambapo maajenti wa AI husitisha utekelezaji kuomba maoni kutoka kwa binadamu kabla ya kuendelea. Hii ni muhimu kwa:

- ✅ **Maamuzi muhimu** - Pata idhini ya binadamu kabla ya kuchukua hatua muhimu
- ✅ **Hali zinazoshindwa kueleweka** - Ruhusu binadamu kufafanua wakati AI haijui wazi
- ✅ **Mapendeleo ya mtumiaji** - Muulize mtumiaji kuchagua kati ya chaguzi mbalimbali
- ✅ **Uzingatiaji na usalama** - Hakikisha ukaguzi wa binadamu kwa shughuli zilizo chini ya kanuni
- ✅ **Uzoefu wa kuingiliana** - Tengeneza maajenti wa mazungumzo yanayojibu maoni ya watumiaji

## 🏗️ Jinsi Inavyofanya Kazi katika Microsoft Agent Framework

Mfumo hutoa vipengele vitatu muhimu kwa HITL:

1. **`RequestInfoExecutor`** - Mtekelezaji maalum anayesitisha mchakato na kutoa `RequestInfoEvent`
2. **`RequestInfoMessage`** - Darasa msingi la maombi yaliyobainishwa yanayotumwa kwa binadamu
3. **`RequestResponse`** - Inahusisha majibu ya binadamu na maombi ya awali kwa kutumia `request_id`

**Mfano wa Mchakato:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Mfano Wetu: Uhifadhi wa Hoteli na Uthibitisho wa Mtumiaji

Tutajenga juu ya mchakato wa hali kwa kuongeza uthibitisho wa binadamu **kabla ya** kupendekeza maeneo mbadala:

1. Mtumiaji anaomba eneo (mfano, "Paris")
2. `availability_agent` anakagua ikiwa vyumba vipo
3. **Kama hakuna vyumba** → `confirmation_agent` huuliza "Je, ungependa kuona mbadala?"
4. Mchakato **husitishwa** kwa kutumia `RequestInfoExecutor`
5. **Binadamu hujibu** "ndiyo" au "hapana" kupitia kuingiza kwenye console
6. `decision_manager` huelekeza kulingana na jibu:
   - **Ndiyo** → Onyesha maeneo mbadala
   - **Hapana** → Ghairi ombi la uhifadhi
7. Onyesha matokeo ya mwisho

Hii inaonyesha jinsi ya kuwapa watumiaji udhibiti juu ya mapendekezo ya maajenti!

---

Tuanze! 🚀


## Hatua ya 1: Ingiza Maktaba Zinazohitajika

Tunaingiza vipengele vya kawaida vya Mfumo wa Wakala pamoja na **madarasa maalum ya mtu-ndani-ya-mzunguko**:
- `RequestInfoExecutor` - Mtekelezaji anayesitisha mtiririko wa kazi kwa ajili ya ingizo la mwanadamu
- `RequestInfoEvent` - Tukio linalotolewa wakati ingizo la mwanadamu linapohitajika
- `RequestInfoMessage` - Darasa msingi kwa mizigo ya maombi yenye aina maalum
- `RequestResponse` - Huuza majibu ya binadamu na maombi
- `WorkflowOutputEvent` - Tukio la kugundua matokeo ya mtiririko wa kazi


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Hatua ya 2: Fafanua Mifano ya Pydantic kwa Matokeo Yaliyo Pangiliwa

Mifano hii hufafanua **schema** ambayo maajenti watarudisha. Tunahifadhi mifano yote kutoka kwenye mtiririko wa masharti na kuongeza:

**Mpya kwa Human-in-the-Loop:**
- `HumanFeedbackRequest` - Sehemu ndogo ya `RequestInfoMessage` inayofafanua mzigo wa ombi unaotumwa kwa watu
  - Inajumuisha `prompt` (swali la kuuliza) na `destination` (muktadha kuhusu jiji lisilopatikana)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Hatua ya 3: Tengeneza Zana ya Kufanya Booking ya Hoteli

Zana ile ile kutoka kwenye mtiririko wa sharti - huchunguza kama vyumba vinapatikana kwenye sehemu unayoelekea.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Hatua ya 4: Fafanua Kazi za Hali kwa Routing

Tunahitaji **kazi nne za hali** kwa mtiririko wetu wa binadamu-ndani ya mzunguko:

**Kutoka kwa mtiririko wa hali:**
1. `has_availability_condition` - Inapeleka wakati hoteli ZIPO disponibles
2. `no_availability_condition` - Inapeleka wakati hoteli HAZIPO disponibles

**Mpya kwa binadamu-ndani ya mzunguko:**
3. `user_wants_alternatives_condition` - Inapeleka wakati mtumiaji anasema "ndiyo" kwa mbadala
4. `user_declines_alternatives_condition` - Inapeleka wakati mtumiaji anasema "hapana" kwa mbadala


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Hatua ya 5: Tengeneza Mtendaji wa Meneja wa Maamuzi

Hii ni **kiini cha mfano wa mtu-ndani-ya-mzunguko**! `DecisionManager` ni `Executor` maalum ambayo:

1. **Inapokea maoni ya binadamu** kupitia vitu vya `RequestResponse`
2. **Inashughulikia uamuzi wa mtumiaji** (ndiyo/hapana)
3. **Inaelekeza mtiririko wa kazi** kwa kutuma ujumbe kwa mawakala husika

Sifa kuu:
- Inatumia mpendwa `@handler` kufichua mbinu kama hatua za mtiririko wa kazi
- Inapokea `RequestResponse[HumanFeedbackRequest, str]` inayojumuisha ombi la awali na jibu la mtumiaji
- Inazalisha ujumbe rahisi wa "ndiyo" au "hapana" unaochochea kazi zetu za hali


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## Hatua 6: Unda Mtekelezaji wa Maonyesho Uliobinafsishwa

Mtekelezaji wa maonyesho uleule kutoka mtiririko wa kazi wa masharti - hutoa matokeo ya mwisho kama pato la mtiririko wa kazi.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Hatua ya 7: Pakia Mabadiliko ya Mazingira

Sanidi mteja wa LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## Hatua ya 8: Tengeneza Maajenti wa AI na Watendaji

Tunatengeneza **vipengele sita vya mtiririko wa kazi**:

**Maajenti (yaliyofungwa ndani ya AgentExecutor):**
1. **availability_agent** - Hukagua upatikana wa hoteli kwa kutumia chombo
2. **confirmation_agent** - 🆕 Huandaa ombi la uthibitisho la binadamu
3. **alternative_agent** - Hupendekeza miji mbadala (wakati mtumiaji anasema ndio)
4. **booking_agent** - Huhimiza uhifadhi (wakati vyumba vinapatikana)
5. **cancellation_agent** - 🆕 Husimamia ujumbe wa kughairi (wakati mtumiaji anasema hapana)

**Watendaji Maalum:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` inayositisha mtiririko wa kazi kwa maingiliano ya binadamu
7. **decision_manager** - 🆕 Mtenzi maalum anayesafirisha kulingana na jibu la binadamu (tayarameshajulikana hapo juu)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Hatua ya 9: Jenga Mtiririko wa Kazi na Binadamu Katika Mzunguko

Sasa tunajenga grafu ya mtiririko wa kazi yenye **muelekeo wa masharti** ikiwa ni pamoja na njia ya binadamu katika mzunguko:

**Muundo wa Mtiririko wa Kazi:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Mikondo Muhimu:**
- `availability_agent → confirmation_agent` (wakati hakuna vyumba)
- `confirmation_agent → prepare_human_request` (badilisha aina)
- `prepare_human_request → request_info_executor` (simama kwa ajili ya binadamu)
- `request_info_executor → decision_manager` (daima - hutoa RequestResponse)
- `decision_manager → alternative_agent` (wakati mtumiaji anasema "ndiyo")
- `decision_manager → cancellation_agent` (wakati mtumiaji anasema "hapana")
- `availability_agent → booking_agent` (wakati vyumba vinapatikana)
- Njia zote zinaisha kwenye `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Hatua ya 10: Endesha Kesi ya Mtihani 1 - Jiji BILA Upatikanaji (Paris na Uthibitisho wa Binadamu)

Mtihani huu unaonyesha **mzunguko kamili wa binadamu katika mchakato**:

1. Omba hoteli Paris
2. wakala wa upatikanaji hacheki → Hakuna vyumba
3. wakala wa uthibitisho huunda swali la uso kwa binadamu
4. mtekelezaji wa maombi ya taarifa **anasitisha mtiririko wa kazi** na hutuma `RequestInfoEvent`
5. **Programu hugundua tukio na kuomba mtumiaji kwenye koni**
6. Mtumiaji anaandika "ndiyo" au "hapana"
7. Programu hutuma majibu kupitia `send_responses_streaming()`
8. meneja wa maamuzi huendesha kulingana na jibu
9. Matokeo ya mwisho yanaonyeshwa

**Mufuniko Muhimu:**
- Tumia `workflow.run_stream()` kwa mzunguko wa kwanza
- Tumia `workflow.send_responses_streaming(pending_responses)` kwa mizunguko inayofuata
- Sikiliza `RequestInfoEvent` kugundua wakati ingizo la binadamu linahitajika
- Sikiliza `WorkflowOutputEvent` kunasa matokeo ya mwisho


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Hatua ya 11: Endesha Kesi ya Mtihani 2 - Jiji KULIKO Upatikanaji (Stockholm - Hakuna Ingizo la Binadamu Linahitajika)

Mtihani huu unaonyesha **njia ya moja kwa moja** wakati vyumba viko available:

1. Omba hotelini Stockholm
2. wakala wa upatikanaji anachunguza → Vyumba vinapatikana ✅
3. wakala wa uhifadhi anapendekeza uhifadhi
4. onesha_matokeo inaonyesha uthibitisho
5. **Hakuna ingizo la binadamu linalohitajika!**

Kazi ya mtiririko inaruka njia ya mtu-kati kabisa wakati vyumba vipo.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Mambo Muhimu na Mbinu Bora za Mtu-Katika-Mzunguko

### ✅ Umejifunza Nini:

#### 1. **Mfano wa RequestInfoExecutor**
Mfano wa mtu-katika-mzunguko katika Muundo wa Wakala wa Microsoft unatumia vipengele vikuu vitatu:
- `RequestInfoExecutor` - Huwia mchakato na kutoa matukio
- `RequestInfoMessage` - Darasa la msingi kwa mizigo ya ombi iliyotambulika (tengeneza darasa chini yake!)
- `RequestResponse` - Huhusisha majibu ya binadamu na maombi ya asili

**Uelewa Muhimu:**
- `RequestInfoExecutor` HAJAKUSANYI pembejeo yenyewe - hubakia tu mchakato
- Msimbo wa programu yako lazima usikilize `RequestInfoEvent` na ukusanye pembejeo
- Lazima uitoe `send_responses_streaming()` ukiwa na kamusi inayoratibu `request_id` na jibu la mtumiaji

#### 2. **Mfano wa Utekelezaji wa Kupitisha Mfululizo**
```python
# Mara ya kwanza ya mzunguko
stream = workflow.run_stream(initial_request)

# Mizunguko inayofuata (baada ya maingizo ya binadamu)
stream = workflow.send_responses_streaming(pending_responses)

# Daima endesha matukio
events = [event async for event in stream]
```

#### 3. **Msingi wa Muktadha wa Tukio**
Sikiliza matukio maalum kudhibiti mchakato:
- `RequestInfoEvent` - Pembejeo ya binadamu inahitajika (mchakato umehusika)
- `WorkflowOutputEvent` - Matokeo ya mwisho yanapatikana (mchakato umekamilika)
- `WorkflowStatusEvent` - Mabadiliko ya hali (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, n.k.)

#### 4. **Watekelezaji Maalum kwa @handler**
`DecisionManager` inaonyesha jinsi ya kuunda watekelezaji ambao:
- Hutumia dekoreta `@handler` kufungua njia kama hatua za mchakato
- Kupokea ujumbe ulioainishwa (mfano, `RequestResponse[HumanFeedbackRequest, str]`)
- Kupitisha mchakato kwa kutuma ujumbe kwa watekelezaji wengine
- Kupata muktadha kupitia `WorkflowContext`

#### 5. **Upangilio Uwazi wa Uamuzi wa Binadamu**
Unaweza kuunda kazi za masharti zinazotathmini majibu ya binadamu:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Matumizi Halisi:

1. **Mchakato wa Idhini**
   - Pata idhini ya meneja kabla ya kushughulikia ripoti za gharama
   - Hitaji kupitia binadamu kabla ya kutuma barua pepe za kiotomatiki
   - Thibitisha miamala yenye thamani kubwa kabla ya utekelezaji

2. **Udhibiti wa Maudhui**
   - Taja maudhui yenye shaka kwa mapitio ya binadamu
   - Waombe wasimamizi kufanya uamuzi wa mwisho katika kesi ngumu
   - Ongeza hali kwa binadamu wakati AI inaaminika kidogo

3. **Huduma kwa Wateja**
   - Acha AI isimamia maswali ya kawaida kiotomatiki
   - Ongeza masuala magumu kwa maajenti wa binadamu
   - Muulize mteja kama anataka kuzungumza na binadamu

4. **Usindikaji Data**
   - Muombe binadamu kutatua upotoshaji wa data
   - Thibitisha tafsiri za AI za nyaraka zisizoeleweka
   - Acha watumiaji wachague kati ya tafsiri chanya kadhaa

5. **Mifumo Muhimu kwa Usalama**
   - Hitaji uthibitisho wa binadamu kabla ya hatua zisizorekebishika
   - Pata idhini kabla ya kupata data nyeti
   - Thibitisha maamuzi katika sekta zilizosimamiwa (huduma za afya, fedha)

6. **Maajenti Shirikishi**
   - Tengeneza roboti za mazungumzo zinazouliza maswali ya ziada
   - Unda wachawi wanaoongoza watumiaji kupitia michakato ngumu
   - Buni maajenti wanaoshirikiana na binadamu hatua kwa hatua

### 🔄 Mlinganisho: Kwa vs Bila Mtu-Katika-Mzunguko

| Kipengele | Mchakato wa Masharti | Mchakato wa Mtu-Katika-Mzunguko |
|---------|---------------------|---------------------------|
| **Utekelezaji** | `workflow.run()` moja | Mzunguko na `run_stream()` + `send_responses_streaming()` |
| **Pembejeo ya Mtumiaji** | Hakuna (kiotomatiki kabisa) | Maandishi ya mwingiliano kupitia `input()` au UI |
| **Vipengele** | Maajenti + Watekelezaji | + RequestInfoExecutor + DecisionManager |
| **Matukio** | AgentExecutorResponse tu | RequestInfoEvent, WorkflowOutputEvent, n.k. |
| **Kuiachia Kazi** | Hakuna kuiachia | Mchakato huwa umeiketisha katika RequestInfoExecutor |
| **Udhibiti wa Binadamu** | Hakuna udhibiti wa binadamu | Watu hufanya maamuzi muhimu |
| **Kesi ya Matumizi** | Uendeshaji | Ushirikiano & ushauri |

### 🚀 Mifano ya Juu:

#### Nafasi Nyingi za Maamuzi ya Binadamu
Unaweza kuwa na nodi nyingi za `RequestInfoExecutor` katika mchakato mmoja:
```python
.add_edge(agent1, request_info_1)  # Uamuzi wa kwanza wa mwanadamu
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Uamuzi wa pili wa mwanadamu
.add_edge(decision_manager_2, final_agent)
```

#### Usimamizi wa Muda wa Kusubiri
Tekeleza muda wa kusubiri majibu ya binadamu:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Chagua chaguo salama kwa chaguo-msingi
```

#### Uunganishaji wa UI Tajiri
Badala ya `input()`, ungana na UI ya wavuti, Slack, Teams, n.k.:
```python
if isinstance(event, RequestInfoEvent):
    # Tuma taarifa kwa njia inayopendekezwa na mtumiaji
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Mtu-Katika-Mzunguko wa Masharti
Uliza pembejeo ya binadamu tu katika hali maalum:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Peleka kwa mwanadamu tu ikiwa ujasiri ni mdogo au thamani ni juu
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Mbinu Bora:

1. **Daima Tengeneza Darasa-Chini ya RequestInfoMessage**
   - Hutoa usalama wa aina na uthibitisho
   - Inawezesha muktadha wa tajiri kwa uonyeshaji wa UI
   - Inaeleza nia ya kila aina ya ombi

2. **Tumia Maelezo ya Kuwahamasisha**
   - Jumuisha muktadha kuhusu unachoomba
   - Elezea matokeo ya kila chaguo
   - Weka maswali kuwa rahisi na wazi

3. **Shughulikia Pembejeo zisizotarajiwa**
   - Thibitisha majibu ya mtumiaji
   - Toa chaguo za msingi kwa pembejeo zisizofaa
   - Toa ujumbe wa makosa wazi

4. **Fuata Nambari za OMBI**
   - Tumia uhusiano kati ya `request_id` na majibu
   - Usijaribu kusimamia hali kwa mikono

5. **Buni kwa Usitake Kuzuia**
   - Usizuie nyuzi kwa kusubiri pembejeo
   - Tumia mifano ya async kote
   - Saidia matukio ya mchakato yanayofanywa kwa wakati mmoja

### 📚 Mafundisho Yanayohusiana:

- **Middleware ya Wakala** - Kata simu za wakala (daftari lililopita)
- **Usimamizi wa Hali ya Mchakato** - Hifadhi hali ya mchakato kati ya utekelezaji
- **Ushirikiano wa Maajenti Wengi** - Changanya mtu-katika-mzunguko na timu za maajenti
- **Misingi ya Muktadha wa Tukio** - Unda mifumo ya kujibua kwa matukio

---

### 🎓 Hongera!

Umejifunza mchakato wa mtu-katika-mzunguko kwa Microsoft Agent Framework! Sasa unajua jinsi ya:
- ✅ Kusimamisha michakato ili kukusanya pembejeo za binadamu
- ✅ Kutumia RequestInfoExecutor na RequestInfoMessage
- ✅ Kushughulikia utekelezaji wa mtiririko kwa matukio
- ✅ Kuunda watekelezaji maalum kwa @handler
- ✅ Kupitisha michakato kulingana na maamuzi ya binadamu
- ✅ Kujenga maajenti wa AI ya mwingiliano wanaoshirikiana na binadamu

**Huu ni mfano muhimu kwa ujenzi wa mifumo ya AI inayoaminika na inayoweza kudhibitiwa!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
